In [ ]:
"""
채용공고 지역 편중 지도 시각화 v2
- 글자 가독성 개선 (흰색 외곽선 stroke)
- 경기/서울 레이블 겹침 해결 (수동 오프셋)
- 0건 지역 레이블 제거
- 컬러바 한글 깨짐 수정
- 전체 디자인 개선

※ 준비물:
   sido.geojson     ← 시도 경계
   seoul_gu.geojson ← 서울 구 경계
   다운로드: https://github.com/southkorea/southkorea-maps (kostat/2018/json/)
"""

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe        # ← 글자 외곽선 핵심
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch
import numpy as np

# ────────────────────────────────────────────────
# 0. 경로 설정
# ────────────────────────────────────────────────
EXCEL_PATH   = r"C:\py_temp\중간프로젝트\채용공고시각화\posting_analysis_table_최종.xlsx"
SIDO_SHP     = r"C:\py_temp\중간프로젝트\채용공고시각화\sido.geojson"
SEOUL_GU_SHP = r"C:\py_temp\중간프로젝트\채용공고시각화\seoul_gu.geojson"
OUTPUT_DIR   = r"C:\py_temp\중간프로젝트\채용공고시각화\output_maps_m3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ────────────────────────────────────────────────
# 1. 한글 폰트 설정  (Windows: 맑은 고딕 자동 감지)
# ────────────────────────────────────────────────
def set_korean_font():
    candidates = [
        'C:/Windows/Fonts/malgun.ttf',              # Windows 맑은 고딕
        'C:/Windows/Fonts/NanumGothic.ttf',
        '/System/Library/Fonts/AppleGothic.ttf',    # macOS
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',  # Linux
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    ]
    for path in candidates:
        if os.path.exists(path):
            fe = fm.FontEntry(fname=path, name='KoreanFont')
            fm.fontManager.ttflist.insert(0, fe)
            plt.rcParams['font.family'] = 'KoreanFont'
            plt.rcParams['axes.unicode_minus'] = False
            print(f"✅ 폰트 설정: {path}")
            return
    # 못 찾으면 matplotlib 기본 한글 폰트 시도
    for name in ['Malgun Gothic', 'NanumGothic', 'AppleGothic']:
        try:
            fm.findfont(fm.FontProperties(family=name), fallback_to_default=False)
            plt.rcParams['font.family'] = name
            plt.rcParams['axes.unicode_minus'] = False
            print(f"✅ 폰트: {name}")
            return
        except Exception:
            pass
    print("⚠ 한글 폰트 미발견 → 한글 깨짐 발생 가능")

set_korean_font()

# ────────────────────────────────────────────────
# 2. 데이터 로드 & 전처리
# ────────────────────────────────────────────────
df = pd.read_excel(EXCEL_PATH)

FOREIGN_KEYWORDS = ['미국', '일본', '베트남', '헝가리', '동경']
mask_foreign = df['시각화용_지역'].apply(
    lambda x: any(k in str(x) for k in FOREIGN_KEYWORDS)
)
df = df[~mask_foreign].copy()
print(f"국내 데이터: {len(df)}건")
print(df['job'].value_counts())

# ────────────────────────────────────────────────
# 3. 집계 함수
# ────────────────────────────────────────────────
def aggregate_sido(df, job_name):
    sub = df[df['job'] == job_name]
    total = len(sub)
    agg = sub.groupby('region_sido').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total * 100).round(1)
    return agg, total

def aggregate_seoul_gu(df, job_name):
    sub = df[(df['job'] == job_name) & (df['region_sido'] == '서울')].copy()
    total = len(sub)
    sub['gu'] = sub['시각화용_지역'].str.replace('^서울 ', '', regex=True).str.strip()
    sub.loc[sub['gu'] == '서울', 'gu'] = '기타'
    agg = sub.groupby('gu').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total * 100).round(1)
    return agg, total

# ────────────────────────────────────────────────
# 4. 색상 팔레트
# ────────────────────────────────────────────────
CMAP_DA = LinearSegmentedColormap.from_list(
    'da_blue', ['#EBF5FB', '#AED6F1', '#5DADE2', '#2874A6', '#1A5276'], N=256)
CMAP_BE = LinearSegmentedColormap.from_list(
    'be_green', ['#E9F7EF', '#A9DFBF', '#52BE80', '#1E8449', '#145A32'], N=256)

JOB_CONFIG = {
    '데이터 분석가': {'cmap': CMAP_DA, 'accent': '#2874A6', 'prefix': 'DA'},
    '백엔드 개발자':  {'cmap': CMAP_BE, 'accent': '#1E8449', 'prefix': 'BE'},
}

# ────────────────────────────────────────────────
# 5. 시도 이름 매핑
# ────────────────────────────────────────────────
SIDO_NAME_MAP = {
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구',
    '인천광역시': '인천', '광주광역시': '광주', '대전광역시': '대전',
    '울산광역시': '울산', '세종특별자치시': '세종',
    '경기도': '경기', '강원특별자치도': '강원', '강원도': '강원',
    '충청북도': '충북', '충청남도': '충남',
    '전라북도': '전북', '전북특별자치도': '전북',
    '전라남도': '전남', '경상북도': '경북', '경상남도': '경남',
    '제주특별자치도': '제주',
    'Seoul': '서울', 'Busan': '부산', 'Daegu': '대구',
    'Incheon': '인천', 'Gwangju': '광주', 'Daejeon': '대전',
    'Ulsan': '울산', 'Sejong': '세종',
    'Gyeonggi-do': '경기', 'Gangwon-do': '강원',
    'Chungcheongbuk-do': '충북', 'Chungcheongnam-do': '충남',
    'Jeollabuk-do': '전북', 'Jeollanam-do': '전남',
    'Gyeongsangbuk-do': '경북', 'Gyeongsangnam-do': '경남',
    'Jeju-do': '제주',
}

# ────────────────────────────────────────────────
# 6. 레이블 수동 오프셋 (겹침 방지)
# ────────────────────────────────────────────────
# (dx, dy) → 경도/위도 단위 오프셋
SIDO_LABEL_OFFSET = {
    '경기': (0.35, -0.10),   # 서울과 겹치지 않도록 오른쪽 아래로
    '서울': (-0.05, 0.10),   # 살짝 위로
    '인천': (-0.30, -0.20),  # 왼쪽 아래로
    '세종': (-0.05,  0.10),  # 살짝 위로
}

SEOUL_GU_LABEL_OFFSET = {
    # 작은 구들 레이블 위치 미세 조정
    '중구':   (0.005, -0.005),
    '용산구': (0.0,  -0.005),
    '종로구': (0.0,   0.005),
    '양천구': (-0.01, -0.01),
}

# ────────────────────────────────────────────────
# 7. 공통: 텍스트 외곽선 효과 (가독성 핵심)
# ────────────────────────────────────────────────
def make_stroke(color='white', linewidth=2.5):
    """흰색(또는 지정색) 외곽선으로 텍스트 가독성을 높임"""
    return [
        pe.withStroke(linewidth=linewidth, foreground=color),
    ]

def text_color_and_stroke(norm_val, threshold=0.45):
    """밝기에 따라 글자색 + 외곽선 색 결정"""
    if norm_val > threshold:
        return 'white', make_stroke('black', 2.0)   # 진한 배경 → 흰 글자 + 검정 외곽
    else:
        return '#1A1A1A', make_stroke('white', 2.5)  # 밝은 배경 → 검정 글자 + 흰 외곽

# ────────────────────────────────────────────────
# 8. 시도별 지도
# ────────────────────────────────────────────────
def draw_sido_map(job_name, sido_gdf_raw, df, output_dir):
    cfg = JOB_CONFIG[job_name]
    agg, total = aggregate_sido(df, job_name)

    # 이름 컬럼 찾기
    name_col = next((c for c in ['name','CTP_KOR_NM','SIDO_NM','kor_name','NAME_1']
                     if c in sido_gdf_raw.columns), None)
    if name_col is None:
        print("⚠ 시도 이름 컬럼 없음:", sido_gdf_raw.columns.tolist()); return

    sido_gdf = sido_gdf_raw.copy()
    sido_gdf['sido_std'] = sido_gdf[name_col].map(SIDO_NAME_MAP).fillna(sido_gdf[name_col])
    sido_gdf = sido_gdf.merge(agg, left_on='sido_std', right_on='region_sido', how='left')
    sido_gdf['count'] = sido_gdf['count'].fillna(0)
    sido_gdf['pct']   = sido_gdf['pct'].fillna(0)

    fig, ax = plt.subplots(figsize=(13, 15), facecolor='#F5F5F5')
    ax.set_facecolor('#F0F4F8')  # 바다 색 (연한 하늘)

    vmax = sido_gdf['count'].max()

    sido_gdf.plot(
        column='count', cmap=cfg['cmap'],
        linewidth=0.7, edgecolor='#777777',
        ax=ax, vmin=0, vmax=vmax,
        missing_kwds={'color': '#DDDDDD'},
    )

    # ── 레이블 ──
    for _, row in sido_gdf.iterrows():
        if row['count'] == 0:
            continue  # 0건은 표시 안 함

        centroid = row.geometry.centroid
        x, y = centroid.x, centroid.y

        # 수동 오프셋 적용
        name = row['sido_std']
        dx, dy = SIDO_LABEL_OFFSET.get(name, (0, 0))
        x += dx; y += dy

        norm_val = row['count'] / vmax if vmax > 0 else 0
        fc, stroke = text_color_and_stroke(norm_val)

        label = f"{name}\n{int(row['count'])}건  ({row['pct']}%)"
        ax.annotate(
            label, xy=(x, y),
            ha='center', va='center',
            fontsize=8.5, fontweight='bold', color=fc,
            path_effects=stroke,
            linespacing=1.5,
        )

    # ── 컬러바 (라벨 한글 깨짐 방지: ylabel 대신 text 사용) ──
    sm = plt.cm.ScalarMappable(cmap=cfg['cmap'],
                                norm=mcolors.Normalize(vmin=0, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.022, pad=0.01, shrink=0.55,
                         aspect=25)
    cbar.ax.tick_params(labelsize=9)
    cbar.ax.set_ylabel('공고 건수', fontsize=10, rotation=270, labelpad=15)

    # ── 제목 박스 ──
    title_str = f"{'데이터 분석가' if job_name=='데이터 분석가' else '백엔드 개발자'} 채용공고 지역 분포"
    sub_str   = f"시·도별  |  총 {total}건"
    ax.set_title(f"{title_str}\n{sub_str}",
                 fontsize=15, fontweight='bold', pad=16,
                 color='#1A1A1A', linespacing=1.8)

    ax.axis('off')
    plt.tight_layout(pad=1.5)

    fpath = os.path.join(output_dir, f"{cfg['prefix']}_sido_map.png")
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#F5F5F5')
    plt.close()
    print(f"저장: {fpath}")


# ────────────────────────────────────────────────
# 9. 서울 구별 지도
# ────────────────────────────────────────────────
def draw_seoul_map(job_name, seoul_gdf_raw, df, output_dir):
    cfg = JOB_CONFIG[job_name]
    agg, total_seoul = aggregate_seoul_gu(df, job_name)

    name_col = next((c for c in ['name','SIG_KOR_NM','GU_NM','kor_name','NAME_2','sggnm']
                     if c in seoul_gdf_raw.columns), None)
    if name_col is None:
        print("⚠ 구 이름 컬럼 없음:", seoul_gdf_raw.columns.tolist()); return

    seoul_gdf = seoul_gdf_raw.copy()
    seoul_gdf['gu_std'] = seoul_gdf[name_col].str.strip()
    seoul_gdf = seoul_gdf.merge(agg, left_on='gu_std', right_on='gu', how='left')
    seoul_gdf['count'] = seoul_gdf['count'].fillna(0)
    seoul_gdf['pct']   = seoul_gdf['pct'].fillna(0)

    fig, ax = plt.subplots(figsize=(15, 13), facecolor='#F5F5F5')
    ax.set_facecolor('#EFF4F9')

    vmax = seoul_gdf['count'].max()

    seoul_gdf.plot(
        column='count', cmap=cfg['cmap'],
        linewidth=0.9, edgecolor='#666666',
        ax=ax, vmin=0, vmax=vmax,
        missing_kwds={'color': '#DDDDDD'},
    )

    # ── 레이블 ──
    for _, row in seoul_gdf.iterrows():
        centroid = row.geometry.centroid
        x, y = centroid.x, centroid.y

        gu_name = row['gu_std']
        dx, dy  = SEOUL_GU_LABEL_OFFSET.get(gu_name, (0, 0))
        x += dx; y += dy

        norm_val = row['count'] / vmax if vmax > 0 else 0
        fc, stroke = text_color_and_stroke(norm_val, threshold=0.38)

        if row['count'] > 0:
            label = f"{gu_name}\n{int(row['count'])}건  ({row['pct']}%)"
            fs = 8.0
        else:
            # 0건: 구 이름만 작게
            label = gu_name
            fs = 7.0
            fc = '#888888'
            stroke = make_stroke('white', 2.0)

        ax.annotate(
            label, xy=(x, y),
            ha='center', va='center',
            fontsize=fs, fontweight='bold', color=fc,
            path_effects=stroke,
            linespacing=1.5,
        )

    # ── 강남구 강조 테두리 ──
    gangnam = seoul_gdf[seoul_gdf['gu_std'] == '강남구']
    if not gangnam.empty:
        gangnam.boundary.plot(ax=ax, linewidth=3.0, edgecolor='#E74C3C', zorder=5)

    # ── 컬러바 ──
    sm = plt.cm.ScalarMappable(cmap=cfg['cmap'],
                                norm=mcolors.Normalize(vmin=0, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.022, pad=0.01, shrink=0.55, aspect=25)
    cbar.ax.tick_params(labelsize=9)
    cbar.ax.set_ylabel('공고 건수', fontsize=10, rotation=270, labelpad=15)

    # ── 제목 ──
    title_str = f"{'데이터 분석가' if job_name=='데이터 분석가' else '백엔드 개발자'} 채용공고 지역 분포"
    sub_str   = f"서울시 구별  |  서울 내 총 {total_seoul}건"
    ax.set_title(f"{title_str}\n{sub_str}",
                 fontsize=15, fontweight='bold', pad=16,
                 color='#1A1A1A', linespacing=1.8)

    # # ── 강남구 주석 (발표용) ──
    # if not gangnam.empty:
    #     cx = gangnam.geometry.centroid.iloc[0].x
    #     cy = gangnam.geometry.centroid.iloc[0].y
    #     row_gn = agg[agg['gu'] == '강남구']
    #     if not row_gn.empty:
    #         gn_pct = row_gn['pct'].values[0]
    #         ax.annotate(
    #             f"강남구\n전체의 {gn_pct}%",
    #             xy=(cx, cy - 0.025),
    #             xytext=(cx + 0.12, cy - 0.09),
    #             fontsize=9, fontweight='bold', color='#E74C3C',
    #             path_effects=make_stroke('white', 2.5),
    #             arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5),
    #             ha='left',
    #         )

    ax.axis('off')
    plt.tight_layout(pad=1.5)

    fpath = os.path.join(output_dir, f"{cfg['prefix']}_seoul_map.png")
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#F5F5F5')
    plt.close()
    print(f"저장: {fpath}")


# ────────────────────────────────────────────────
# 10. 실행
# ────────────────────────────────────────────────
def load_geo(path):
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf

def main():
    for path, label in [(SIDO_SHP, '시도'), (SEOUL_GU_SHP, '서울 구')]:
        if not os.path.exists(path):
            print(f"[ERROR] {label} 파일 없음: {path}")
            print("cmd에서 실행:")
            if label == '시도':
                print('  curl -L -o sido.geojson "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-provinces-2018-geo.json"')
            else:
                print('  curl -L -o seoul_gu.geojson "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-municipalities-2018-geo.json"')
            return

    print("시도 로드...")
    sido_gdf = load_geo(SIDO_SHP)
    print(f"  컬럼: {sido_gdf.columns.tolist()}")

    print("서울 구 로드...")
    gu_all = load_geo(SEOUL_GU_SHP)
    print(f"  컬럼: {gu_all.columns.tolist()}")

    # 서울 구 필터
    name_col = next((c for c in ['name','SIG_KOR_NM','GU_NM','sggnm','NAME_2']
                     if c in gu_all.columns), None)
    SEOUL_GU_LIST = [
        '강남구','강동구','강북구','강서구','관악구','광진구','구로구','금천구',
        '노원구','도봉구','동대문구','동작구','마포구','서대문구','서초구',
        '성동구','성북구','송파구','양천구','영등포구','용산구','은평구',
        '종로구','중구','중랑구'
    ]
    if 'CTPRVN_CD' in gu_all.columns:
        seoul_gu = gu_all[gu_all['CTPRVN_CD'] == '11'].copy()
    elif 'code' in gu_all.columns:
        seoul_gu = gu_all[gu_all['code'].astype(str).str.startswith('11')].copy()
    elif name_col:
        seoul_gu = gu_all[gu_all[name_col].isin(SEOUL_GU_LIST)].copy()
    else:
        seoul_gu = gu_all
    print(f"  서울 구 수: {len(seoul_gu)}")

    for job in ['데이터 분석가', '백엔드 개발자']:
        print(f"\n{'='*40}\n{job} 지도 생성\n{'='*40}")
        draw_sido_map(job, sido_gdf, df, OUTPUT_DIR)
        draw_seoul_map(job, seoul_gu, df, OUTPUT_DIR)

    print(f"\n✅ 완료! → {os.path.abspath(OUTPUT_DIR)}/")

if __name__ == "__main__":
    main()


✅ 폰트 설정: C:/Windows/Fonts/malgun.ttf
국내 데이터: 877건
job
백엔드 개발자    603
데이터 분석가    274
Name: count, dtype: int64
시도 로드...
  컬럼: ['name', 'base_year', 'name_eng', 'code', 'geometry']
서울 구 로드...
  컬럼: ['name', 'base_year', 'name_eng', 'code', 'geometry']
  서울 구 수: 25

데이터 분석가 지도 생성
저장: C:\py_temp\중간프로젝트\output_maps_m3\DA_sido_map.png
저장: C:\py_temp\중간프로젝트\output_maps_m3\DA_seoul_map.png

백엔드 개발자 지도 생성
저장: C:\py_temp\중간프로젝트\output_maps_m3\BE_sido_map.png
저장: C:\py_temp\중간프로젝트\output_maps_m3\BE_seoul_map.png

✅ 완료! → C:\py_temp\중간프로젝트\output_maps_m3/
